In [1]:
import sympy as sp
import numpy as np

### Base SIRS

In [6]:
# 1. Define the Symbols
beta, gamma, theta, mu = sp.symbols('beta gamma theta mu', positive=True, real=True)
S, I, R = sp.symbols('S I R', real=True)

# 2. Define the 'Full' SIRS Model (with Vital Dynamics mu)
eq_full_S = mu - beta * S * I - mu * S + theta * R
eq_full_I = beta * S * I - (gamma + mu) * I
eq_full_R = gamma * I - (theta + mu) * R

# 3. Define the 'Reduced' SIRS Model (mu = 0)
eq_red_S = theta * R - beta * S * I
eq_red_I = beta * S * I - gamma * I
eq_red_R = gamma * I - theta * R

In [7]:
# Solve for Endemic Equilibrium (We want the non-zero I solution)
# We assume I != 0
sol_full = sp.solve([eq_full_S, eq_full_I, eq_full_R], [S, I, R], dict=True)

eq_pop = S + I + R - 1
sol_red = sp.solve([eq_red_S, eq_red_I, eq_red_R, eq_pop], [S, I, R], dict=True)

In [8]:
# Filter for the solution where I is not 0 (the endemic state)
# In this specific system, usually the second solution is the endemic one
EE_full = None
for sol in sol_full:
    if sol[I] != 0:
        EE_full = sp.Matrix([sol[S], sol[I], sol[R]])
        break


EE_red = None
for sol in sol_red:
    if sol[I] != 0:
        EE_red = sp.Matrix([sol[S], sol[I], sol[R]])
        break

In [13]:
# 4. Calculate the Exact Euclidean Distance
# Vector difference
diff_vector = EE_full - EE_red

# Calculate the Norm (Euclidean distance)
# simplify() is used to clean up the algebraic mess
dist_squared = sp.simplify(diff_vector.dot(diff_vector))
exact_distance = sp.sqrt(dist_squared)

print("--- Exact Euclidean Distance (Symbolic) ---")
# sp.pprint prints it in a nice mathematical format
sp.pprint(exact_distance)

--- Exact Euclidean Distance (Symbolic) ---
   ___________________________________________________________________________ ↪
  ╱  2                                             2    2        2             ↪
╲╱  γ ⋅((β - γ)⋅(γ + μ + θ) + (γ + θ)⋅(-β + γ + μ))  + μ ⋅(γ + θ) ⋅(γ + μ + θ) ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                          β⋅(γ + θ)⋅(γ + μ + θ ↪

↪ ____________________________________________________________
↪ 2                                                         2 
↪   + (θ⋅(β - γ)⋅(γ + μ + θ) + (γ + θ)⋅(μ + θ)⋅(-β + γ + μ))  
↪ ────────────────────────────────────────────────────────────
↪ )                                                           


In [12]:

# 5. Verification: Series Expansion (Taylor Series)
# We expand the distance around mu = 0 to see the linear behavior
print("\n--- Series Expansion around mu = 0 (First Order) ---")
approx_distance = sp.series(exact_distance, mu, 0, 2).removeO() # Order 2 gives us the linear term
sp.pprint(sp.simplify(approx_distance))


--- Series Expansion around mu = 0 (First Order) ---
      __________________________________________________
     ╱                                                2 
    ╱   2        2          4   ⎛        2          2⎞  
μ⋅╲╱   γ ⋅(β + θ)  + (γ + θ)  + ⎝-β⋅γ + γ  + γ⋅θ + θ ⎠  
────────────────────────────────────────────────────────
                                2                       
                       β⋅(γ + θ)                        


### No Memory

In [4]:
# 1. Define the Symbols
beta0, gamma, theta, mu, alpha = sp.symbols('beta0 gamma theta mu alpha', positive=True, real=True)
I, R = sp.symbols('I R', real=True)

# 2. Define the full behavioural SIRS Model
eq_full_I = beta0 / (1 + alpha*I) * (1-R-I) * I - (gamma + mu) * I
eq_full_R = gamma * I - (theta + mu) * R

# 3. Define the reduced behavioural SIRS Model
eq_red_I = beta0 / (1 + alpha * I) * (1-R-I) * I - gamma * I
eq_red_R = gamma * I - theta * R

In [5]:
# Solve for Endemic Equilibrium (We want the non-zero I solution)
sol_full = sp.solve([eq_full_I, eq_full_R], [I, R], dict=True)

sol_red = sp.solve([eq_red_I, eq_red_R], [I, R], dict=True)

In [6]:
# Filter for the solution where I is not 0 (the endemic state)
# In this specific system, usually the second solution is the endemic one
EE_full = None
for sol in sol_full:
    if sol[I] != 0:
        EE_full = sp.Matrix([sol[I], sol[R]])
        break


EE_red = None
for sol in sol_red:
    if sol[I] != 0:
        EE_red = sp.Matrix([sol[I], sol[R]])
        break

In [8]:
# 4. Calculate the Exact Euclidean Distance
# Vector difference
diff_vector = EE_full - EE_red

# Calculate the Norm (Euclidean distance)
# simplify() is used to clean up the algebraic mess
dist_squared = sp.simplify(diff_vector.dot(diff_vector))
exact_distance = sp.sqrt(dist_squared)

print("--- Exact Euclidean Distance (Symbolic) ---")
# sp.pprint prints it in a nice mathematical format
print(exact_distance)

--- Exact Euclidean Distance (Symbolic) ---
sqrt(gamma**2*((beta0 - gamma)*(alpha*gamma*mu + alpha*gamma*theta + alpha*mu**2 + alpha*mu*theta + beta0*gamma + beta0*mu + beta0*theta) + (-beta0 + gamma + mu)*(alpha*gamma*theta + beta0*gamma + beta0*theta))**2 + (theta*(beta0 - gamma)*(alpha*gamma*mu + alpha*gamma*theta + alpha*mu**2 + alpha*mu*theta + beta0*gamma + beta0*mu + beta0*theta) + (mu + theta)*(-beta0 + gamma + mu)*(alpha*gamma*theta + beta0*gamma + beta0*theta))**2)/((alpha*gamma*theta + beta0*gamma + beta0*theta)*(alpha*gamma*mu + alpha*gamma*theta + alpha*mu**2 + alpha*mu*theta + beta0*gamma + beta0*mu + beta0*theta))


In [10]:
# Hardcoded I_e, R_e
beta0, gamma, theta, mu, alpha, R0 = sp.symbols('beta0 gamma theta mu alpha, R0', positive=True, real=True)
If, Rf, Sf, Mf = sp.symbols('If Rf Sf Mf', real = True)
Ir, Rr, Sr, Mr = sp.symbols('Ir Rr Sr Mr', real = True)

full = {If: (beta0/(mu + gamma)-1) / (beta0/(mu + gamma)*(1+gamma / (mu + theta) + alpha) ),
 Rf: (gamma / (mu + theta) * (beta0/(mu + gamma)-1) / (beta0/(mu + gamma)*(1+gamma / (mu + theta) ) + alpha)),
 Sf: (mu + gamma) / beta0,
 Mf: (beta0/(mu + gamma)-1) / (beta0/(mu + gamma)*(1+gamma / (mu + theta) + alpha) )}

red = {Ir: (beta0/gamma-1) / (beta0/gamma*(1+gamma / theta + alpha)),
 Rr: (gamma / theta * (beta0/gamma-1) / (beta0/gamma*(1+gamma / theta + alpha))),
 Sr: gamma / beta0,
 Mr: (beta0/gamma-1) / (beta0/gamma*(1+gamma / theta + alpha))}

diff = sp.Matrix(list(full.values())) - sp.Matrix(list(red.values()))

dist_squared = sp.simplify(diff.dot(diff))
exact_distance = sp.sqrt(dist_squared)


In [11]:
print("--- Exact Euclidean Distance (Symbolic) ---")
# sp.pprint prints it in a nice mathematical format
sp.pprint(exact_distance)

--- Exact Euclidean Distance (Symbolic) ---
           ___________________________________________________________________ ↪
          ╱                                                         2          ↪
         ╱                    ⎛    ⎛β₀    ⎞             ⎛ β₀      ⎞⎞    ⎛      ↪
        ╱               2     ⎜  γ⋅⎜── - 1⎟     (γ + μ)⋅⎜───── - 1⎟⎟    ⎜      ↪
       ╱    ⎛γ    γ + μ⎞      ⎜    ⎝γ     ⎠             ⎝γ + μ    ⎠⎟    ⎜      ↪
      ╱     ⎜── - ─────⎟  + 2⋅⎜────────────── - ───────────────────⎟  + ⎜───── ↪
     ╱      ⎝β₀    β₀  ⎠      ⎜   ⎛    γ    ⎞      ⎛      γ      ⎞ ⎟    ⎜⎛     ↪
    ╱                         ⎜β₀⋅⎜α + ─ + 1⎟   β₀⋅⎜α + ───── + 1⎟ ⎟    ⎜⎜     ↪
   ╱                          ⎝   ⎝    θ    ⎠      ⎝    μ + θ    ⎠ ⎠    ⎜⎜     ↪
  ╱                                                                     ⎜⎜α +  ↪
╲╱                                                                      ⎝⎝     ↪

↪ _____________________________________________
↪               

### One Memory layer

In [13]:
# Define the system
beta0, gamma, theta, mu, alpha = sp.symbols('beta0 gamma theta mu alpha', positive=True, real=True)
S, I, R, M = sp.symbols('S I R M', positive=True, real=True)

In [19]:
dotS = mu*(1-S) - beta0/(1 + alpha*M)*I*S + theta*R
dotI = I*(beta0/(1+alpha*M)*S - (mu + gamma))
dotR = gamma*I - (mu + theta)*R
dotM = alpha*(I-M)

In [20]:
sol = sp.solve([dotS, dotI, dotR, dotM], [S, I, R, M], dict=True)

In [21]:
sol

[{I: -(mu + theta)*(-beta0 + gamma + mu)/(alpha*gamma*mu + alpha*gamma*theta + alpha*mu**2 + alpha*mu*theta + beta0*gamma + beta0*mu + beta0*theta),
  M: (beta0*mu + beta0*theta - gamma*mu - gamma*theta - mu**2 - mu*theta)/(alpha*gamma*mu + alpha*gamma*theta + alpha*mu**2 + alpha*mu*theta + beta0*gamma + beta0*mu + beta0*theta),
  R: gamma*(beta0 - gamma - mu)/(alpha*gamma*mu + alpha*gamma*theta + alpha*mu**2 + alpha*mu*theta + beta0*gamma + beta0*mu + beta0*theta),
  S: (gamma + mu)*(alpha*mu + alpha*theta + gamma + mu + theta)/(alpha*gamma*mu + alpha*gamma*theta + alpha*mu**2 + alpha*mu*theta + beta0*gamma + beta0*mu + beta0*theta)}]